# Driving Datasets and 3D Detection Metrics

**Module 01 · Perception Systems** · Notebook 8 of 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/01_Perception/08_driving_datasets.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~60 min | [04 Object detection](04_object_detection.ipynb) | ISO/PAS 8800:2024; ISO 21448:2022; ISO 34503:2023 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Describe the major public driving datasets (KITTI, nuScenes, Waymo Open, BDD100K, Argoverse 2) and recent European datasets
2. Read dataset statistics critically: scenes vs frames vs annotated keyframes, sensor setups, licences
3. Analyse class imbalance and the long tail, and explain their safety implications
4. Compute nuScenes-style 3D detection metrics (centre-distance AP, mAP, TP errors, NDS) from scratch
5. Select datasets for a given ODD and identify coverage gaps
6. Relate dataset properties to ISO/PAS 8800 data requirements and ISO 21448 triggering conditions

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Wedge, Patch
from IPython.display import display

rng = np.random.default_rng(42)
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print(f"NumPy {np.__version__}, pandas {pd.__version__}")

---
## 1. Overview of major public datasets

Perception models need **large, diverse and accurately labelled** data. Beware of three different counts: **scenes/sequences**, **sensor frames** (often at 10–20 Hz) and **annotated keyframes**.

| Dataset | Year / venue | Scenes | Annotated data | Sensors | 3D boxes | Notes |
|---|---|---|---|---|---|---|
| **KITTI** | 2012, CVPR | — | 7,481 training + 7,518 test images (object detection) | 2 colour + 2 greyscale cameras, 1 LiDAR (HDL-64E), GPS/IMU | Yes | KIT & Toyota Technological Institute at Chicago; Karlsruhe; CC BY-NC-SA 3.0 |
| **nuScenes** | 2019 (arXiv), CVPR 2020 | 1,000 × 20 s | 1.4 M camera images; ~40 K annotated keyframes (2 Hz) | 6 cameras, 5 radars, 1 LiDAR (32-beam) | Yes, 23 classes | Boston & Singapore; non-commercial terms of use |
| **Waymo Open (Perception)** | CVPR 2020 | 1,150 × 20 s (v1; later expanded) | ~230 K frames (10 Hz) | 5 cameras, 5 LiDARs (1 mid-range + 4 short-range) | Yes | Also Motion and End-to-End datasets; non-commercial licence |
| **BDD100K** | 2018 (arXiv), CVPR 2020 | 100 K videos (~40 s) | 100 K annotated keyframe images | Camera (+GPS/IMU) | No (2D only) | NYC, SF Bay Area and other US regions; weather/time diversity |
| **Argoverse 2 (Sensor)** | NeurIPS 2021 Datasets & Benchmarks | 1,000 sequences | 3D cuboids, 26 evaluated categories | 7 ring + 2 stereo cameras, 2 LiDARs | Yes | HD maps; supersedes Argoverse 1; CC BY-NC-SA 4.0 |
| **Zenseact Open Dataset (ZOD)** | ICCV 2023 | 1,473 sequences × 20 s; 29 drives | 100 K curated frames | Camera, LiDAR, GNSS/IMU | Yes (up to 245 m) | Collected over two years in several European countries |
| **MAN TruckScenes** | NeurIPS 2024 Datasets & Benchmarks | — | — | 4 cameras, 6 LiDARs, 6 radars (4D, 360°) | Yes | Trucks; first dataset with 360° 4D radar |

Further European datasets: **A2D2** (Audi, Germany; Geyer et al. 2020) and **View-of-Delft** (4D radar; Palffy et al. 2022). Always check the current licence before use — most of these datasets are licensed for **non-commercial** use only.

In [ ]:
datasets = ['KITTI', 'nuScenes', 'Waymo Open', 'BDD100K', 'Argoverse 2', 'ZOD', 'MAN TruckScenes']
sensors = pd.DataFrame({
    'Cameras': [4, 6, 5, 1, 9, None, 4],
    'LiDARs':  [1, 1, 5, 0, 2, None, 6],
    'Radars':  [0, 5, 0, 0, 0, None, 6],
}, index=datasets)
annotated = pd.Series({'KITTI': 7_481, 'nuScenes': 40_000, 'Waymo Open': 230_000, 'BDD100K': 100_000, 'ZOD': 100_000},
                      name='annotated frames (approx.)')

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
s = sensors.dropna()
xx = np.arange(len(s)); w = 0.27
for k, (col, c) in enumerate(zip(['Cameras', 'LiDARs', 'Radars'], [OKABE_ITO[0], OKABE_ITO[2], OKABE_ITO[4]])):
    axes[0].bar(xx + (k - 1) * w, s[col], w, label=col, color=c)
axes[0].set_xticks(xx); axes[0].set_xticklabels(s.index, rotation=20)
axes[0].set_ylabel('Number of sensors'); axes[0].set_title('Sensor setups'); axes[0].legend()
axes[1].barh(annotated.index, annotated.values, color=OKABE_ITO[4])
axes[1].set_xscale('log'); axes[1].set_xlabel('Annotated frames (log scale, approximate)')
axes[1].set_title('Annotated frames (KITTI: training split; nuScenes: keyframes; Waymo: v1 frames)')
for i, v in enumerate(annotated.values):
    axes[1].text(v * 1.1, i, f'{v/1e3:.0f}K', va='center')
plt.tight_layout(); plt.show()

print("--- Dataset statistics (text summary) ---")
for d in s.index:
    print(f"  {d}: {int(s.loc[d,'Cameras'])} cameras, {int(s.loc[d,'LiDARs'])} LiDARs, {int(s.loc[d,'Radars'])} radars")
for d, v in annotated.items():
    print(f"  {d}: ~{v:,} annotated frames")
print(f"  Only {', '.join(s.index[s['Radars'] > 0])} include radar; BDD100K has no 3D data.")

---
## 2. KITTI deep dive

**KITTI Vision Benchmark Suite** (Geiger, Lenz & Urtasun, CVPR 2012) — Karlsruhe Institute of Technology and Toyota Technological Institute at Chicago; licence CC BY-NC-SA 3.0 (non-commercial).

- **Location:** Karlsruhe, Germany (city, rural roads, motorway); daytime, mostly fair weather.
- **Sensors:** 2 colour + 2 greyscale cameras (stereo), Velodyne HDL-64E LiDAR, GPS/IMU.
- **Tasks:** stereo, optical flow, odometry, 2D/3D object detection, tracking, depth, segmentation.
- **Classes:** 8 annotated classes plus DontCare; the detection benchmark officially evaluates **3**: Car, Pedestrian, Cyclist (3D IoU threshold 0.7 for cars, 0.5 for pedestrians and cyclists).

In [ ]:
# KITTI object training labels (7,481 images), excluding DontCare
kitti_classes = {'Car': 28742, 'Van': 2914, 'Truck': 1094, 'Pedestrian': 4487, 'Person_sitting': 222,
                 'Cyclist': 1627, 'Tram': 511, 'Misc': 973}
ser = pd.Series(kitti_classes).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(11, 5))
colours = [OKABE_ITO[5] if c in ('Pedestrian', 'Person_sitting', 'Cyclist') else OKABE_ITO[4] for c in ser.index]
bars = ax.bar(ser.index, ser.values, color=colours)
for b in bars:
    ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f'{int(b.get_height()):,}', ha='center', va='bottom', fontsize=9)
ax.set_yscale('log'); ax.set_ylabel('Number of instances (log scale)'); ax.set_xlabel('Class')
ax.set_title('KITTI object training labels by class (orange = vulnerable road users)')
ax.legend(handles=[Patch(color=OKABE_ITO[5], label='VRU classes'), Patch(color=OKABE_ITO[4], label='Other')])
plt.tight_layout(); plt.show()

total = ser.sum(); vru = ser[['Pedestrian', 'Person_sitting', 'Cyclist']].sum()
print("--- KITTI class distribution (text summary) ---")
for c, n in ser.items():
    print(f"  {c:15s} {n:6,} ({100*n/total:4.1f} %)")
print(f"  Most common: {ser.index[0]} ({ser.iloc[0]:,}); least common: {ser.index[-1]} ({ser.iloc[-1]:,}); "
      f"imbalance ratio {ser.iloc[0]/ser.iloc[-1]:.0f}:1")
print(f"  VRU classes make up {100*vru/total:.1f} % of labels - yet they dominate the safety risk.")

### A KITTI-style annotated image

We cannot download KITTI here (registration and licence acceptance are required), so we draw a synthetic scene in the KITTI image format (1242 × 375 px) with 2D box annotations.

In [ ]:
W, H = 1242, 375
fig, ax = plt.subplots(figsize=(15, 5))
ax.add_patch(Rectangle((0, 0), W, H / 3, color='#BFDDF2'))           # sky
ax.add_patch(Rectangle((0, H / 3), W, 2 * H / 3, color='#7A7A7A'))   # road
ax.add_patch(Rectangle((50, 50), 150, H / 3 - 50, color='#A8A8A8')); ax.add_patch(Rectangle((900, 70), 200, H / 3 - 70, color='#8C8C8C'))
for x in range(0, W, 80):
    ax.add_patch(Rectangle((x, H - 50), 40, 5, color='white'))
objects = [('Car', (400, 180, 200, 120)), ('Car', (700, 200, 180, 100)),
           ('Pedestrian', (250, 200, 50, 120)), ('Cyclist', (550, 210, 60, 100))]
col = {'Car': OKABE_ITO[4], 'Pedestrian': OKABE_ITO[5], 'Cyclist': OKABE_ITO[0]}
for cls, (x, y, w, h) in objects:
    ax.add_patch(Rectangle((x, y), w, h, color='#505050'))
    ax.add_patch(Rectangle((x, y), w, h, fill=False, lw=3, edgecolor=col[cls]))
    ax.text(x, y - 8, cls, color=col[cls], fontsize=11, fontweight='bold')
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect('equal'); ax.grid(False)
ax.set_xlabel('u (px)'); ax.set_ylabel('v (px)')
ax.set_title('Synthetic KITTI-style scene with 2D box annotations (illustrative)')
ax.legend(handles=[Patch(color=c, label=k) for k, c in col.items()], loc='upper right')
plt.tight_layout(); plt.show()
print("--- Annotations in the synthetic scene ---")
for i, (cls, b) in enumerate(objects, 1):
    print(f"  {i}. {cls}: [x, y, w, h] = {list(b)}")
print("Real KITTI labels also contain 3D size, location, rotation_y, truncation and occlusion level per object.")

---
## 3. nuScenes deep dive

**nuScenes** (Caesar et al., CVPR 2020) — Motional (formerly nuTonomy). Improvements over KITTI:
- **Full 360° coverage** with 6 cameras, plus **5 radars** and a 32-beam roof LiDAR (20 Hz); cameras at 12 Hz
- **Boston and Singapore** (left- and right-hand traffic), including rain and night
- 1,000 scenes of 20 s; annotated keyframes at 2 Hz; **23 classes**, 8 attributes; the detection benchmark uses **10 classes**
- Sensor synchronisation: each camera is triggered when the top LiDAR sweeps across the centre of its field of view

The sketch below shows the approximate camera orientations and fields of view (70° for five cameras, 110° for the rear camera).

In [ ]:
cams = {'CAM_FRONT': (0, 70), 'CAM_FRONT_LEFT': (55, 70), 'CAM_FRONT_RIGHT': (-55, 70),
        'CAM_BACK_LEFT': (110, 70), 'CAM_BACK_RIGHT': (-110, 70), 'CAM_BACK': (180, 110)}
radars = ['RADAR_FRONT', 'RADAR_FRONT_LEFT', 'RADAR_FRONT_RIGHT', 'RADAR_BACK_LEFT', 'RADAR_BACK_RIGHT']
fig, ax = plt.subplots(figsize=(8, 8))
ax.add_patch(Rectangle((-1, -2.3), 2, 4.6, color='lightgrey', ec='black', lw=1.5))
for k, (name, (yaw, fov)) in enumerate(cams.items()):
    th = yaw + 90                                   # plot: forward = +y
    ax.add_patch(Wedge((0, 0.5), 6, th - fov / 2, th + fov / 2, alpha=0.25, color=OKABE_ITO[k % 7]))
    ax.text(6.6 * np.cos(np.deg2rad(th)), 0.5 + 6.6 * np.sin(np.deg2rad(th)), name.replace('CAM_', ''),
            ha='center', va='center', fontsize=8)
ax.plot(0, 0.5, '^', color=OKABE_ITO[2], ms=14, label='LIDAR_TOP (360°)')
ax.set_xlim(-8, 8); ax.set_ylim(-8, 8); ax.set_aspect('equal')
ax.set_xlabel('Lateral (m, schematic)'); ax.set_ylabel('Longitudinal (m, schematic)')
ax.set_title('nuScenes camera fields of view (schematic, approximate orientations)'); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

covered = np.zeros(360, bool)
for yaw, fov in cams.values():
    a = (np.arange(int(yaw - fov / 2), int(yaw + fov / 2)) % 360); covered[a] = True
print(f"Cameras: {', '.join(cams)}; radars: {', '.join(radars)}; LiDAR: LIDAR_TOP (32 beams)")
print(f"Azimuth covered by at least one camera in this sketch: {covered.sum()} of 360 degrees")

**Detection-benchmark classes (10):** car, truck, bus, trailer, construction vehicle, pedestrian, motorcycle, bicycle, traffic cone, barrier. Class counts are best computed directly with the official `nuscenes-devkit` (the optional cell below shows how; it requires registration and a download of the `v1.0-mini` split).

In [ ]:
# Optional: load real nuScenes data (requires registration at nuscenes.org and the v1.0-mini download).
# !pip install nuscenes-devkit
# from nuscenes.nuscenes import NuScenes
# nusc = NuScenes(version='v1.0-mini', dataroot='/content/nuscenes', verbose=True)
# from collections import Counter
# counts = Counter(a['category_name'] for a in nusc.sample_annotation)
# print(counts.most_common(10))
# nusc.render_sample(nusc.sample[0]['token'])
print("Optional real-data cell (commented out): uses nuscenes-devkit on v1.0-mini.")

---
## 4. 3D detection metrics: nuScenes mAP and NDS

2D metrics (notebook 04) use box IoU. For 3D detection, nuScenes defines:

- **AP by centre distance:** a detection matches a GT object of the same class if their **BEV centre distance** is below a threshold $d \in \{0.5, 1, 2, 4\}$ m. AP is computed per class and threshold (ignoring the low-recall/low-precision corners: recall and precision below 10 % are clipped) and averaged → **mAP**.
- **True-positive error metrics** (at $d = 2$ m): translation (mATE, m), scale (mASE, 1 − IoU after alignment), orientation (mAOE, rad), velocity (mAVE, m/s), attribute (mAAE, 1 − accuracy).
- **nuScenes Detection Score:** $\mathrm{NDS} = \frac{1}{10}\left[5\,\mathrm{mAP} + \sum_{\mathrm{mTP}} \left(1 - \min(1, \mathrm{mTP})\right)\right]$.

Below we implement centre-distance AP and NDS from scratch on a synthetic BEV detection set and show how performance falls with range. (The AP here uses the standard all-point interpolation without nuScenes' 10 % clipping, to keep the code short.)

> **Illustrative data:** values below are synthetic, chosen for teaching; they are not measurements of a real detector.

In [ ]:
def simulate_bev(n_frames=400, seed=3):
    r = np.random.default_rng(seed)
    gts, dets = [], []
    for f in range(n_frames):
        n = r.integers(3, 9)
        rng_ = r.uniform(5, 70, n); az = r.uniform(-np.pi / 3, np.pi / 3, n)
        xy = np.stack([rng_ * np.cos(az), rng_ * np.sin(az)], 1)
        for k in range(n):
            gts.append((f, xy[k], rng_[k]))
            p_det = 0.97 - 0.008 * max(rng_[k] - 20, 0)                # detection rate falls beyond 20 m
            if r.random() < p_det:
                sigma = 0.15 + 0.012 * rng_[k]                           # localisation error grows with range
                score = float(np.clip(r.normal(0.8 - 0.006 * rng_[k], 0.12), 0.01, 0.99))
                dets.append((f, xy[k] + r.normal(0, sigma, 2), score))
        for _ in range(r.poisson(0.7)):
            rr, aa = r.uniform(5, 70), r.uniform(-np.pi / 3, np.pi / 3)
            dets.append((f, np.array([rr * np.cos(aa), rr * np.sin(aa)]), float(np.clip(r.normal(0.35, 0.12), 0.01, 0.99))))
    return gts, dets

def ap_centre_distance(gts, dets, thr):
    """All-point interpolated AP with greedy matching on BEV centre distance (one class)."""
    by_frame = {}
    for i, (f, xy, _) in enumerate(gts):
        by_frame.setdefault(f, []).append(i)
    used = np.zeros(len(gts), bool)
    order = sorted(range(len(dets)), key=lambda i: -dets[i][2])
    tp = np.zeros(len(order), bool); err = []
    for n, i in enumerate(order):
        f, xy, _ = dets[i]
        cand = [j for j in by_frame.get(f, []) if not used[j]]
        if not cand: continue
        d = [np.linalg.norm(gts[j][1] - xy) for j in cand]
        k = int(np.argmin(d))
        if d[k] < thr:
            tp[n] = True; used[cand[k]] = True; err.append(d[k])
    ctp = np.cumsum(tp); rec = ctp / len(gts); prec = ctp / np.arange(1, len(tp) + 1)
    r_ = np.concatenate([[0], rec, [1]]); p_ = np.concatenate([[0], prec, [0]])
    p_ = np.maximum.accumulate(p_[::-1])[::-1]
    idx = np.where(r_[1:] != r_[:-1])[0]
    return float(np.sum((r_[idx + 1] - r_[idx]) * p_[idx + 1])), (np.mean(err) if err else np.nan)

gts, dets = simulate_bev()
thrs = [0.5, 1.0, 2.0, 4.0]
aps, ate = [], None
for t in thrs:
    ap, e = ap_centre_distance(gts, dets, t)
    aps.append(ap)
    if t == 2.0: ate = e
mAP = float(np.mean(aps))
print(f"Synthetic set: {len(gts)} GT objects, {len(dets)} detections (single class)")
for t, a in zip(thrs, aps):
    print(f"  AP @ {t} m centre distance: {a:.3f}")
print(f"  mAP (mean over thresholds) = {mAP:.3f}; mean translation error of TPs at 2 m (ATE) = {ate:.2f} m")

# NDS with the computed mAP/ATE and illustrative values for the other TP errors
tp_errors = {'mATE (m)': ate, 'mASE': 0.28, 'mAOE (rad)': 0.45, 'mAVE (m/s)': 0.60, 'mAAE': 0.20}
nds = 0.1 * (5 * mAP + sum(1 - min(1.0, v) for v in tp_errors.values()))
print("  TP errors (mATE computed here; the others illustrative): " + ", ".join(f"{k} = {v:.2f}" for k, v in tp_errors.items()))
print(f"  NDS = 0.1 * [5 * {mAP:.3f} + sum(1 - min(1, mTP))] = {nds:.3f}")

In [ ]:
bins = [(5, 20), (20, 40), (40, 70)]
rows = []
for lo, hi in bins:
    g_sub = [g for g in gts if lo <= g[2] < hi]
    d_sub = [d for d in dets if lo <= np.linalg.norm(d[1]) < hi]
    ap1, e1 = ap_centre_distance(g_sub, d_sub, 1.0)
    ap2, _ = ap_centre_distance(g_sub, d_sub, 2.0)
    rows.append({'range (m)': f'{lo}-{hi}', 'GT objects': len(g_sub), 'AP@1m': round(ap1, 3), 'AP@2m': round(ap2, 3),
                 'mean centre error (m)': round(e1, 2)})
rdf = pd.DataFrame(rows); display(rdf)
fig, ax = plt.subplots(figsize=(9, 4.5))
xx = np.arange(len(rdf)); w = 0.35
ax.bar(xx - w / 2, rdf['AP@1m'], w, label='AP @ 1 m', color=OKABE_ITO[4])
ax.bar(xx + w / 2, rdf['AP@2m'], w, label='AP @ 2 m', color=OKABE_ITO[0])
ax.set_xticks(xx); ax.set_xticklabels(rdf['range (m)']); ax.set_xlabel('Range bin (m)'); ax.set_ylabel('AP')
ax.set_ylim(0, 1.05); ax.set_title('Centre-distance AP by range (synthetic detector, illustrative)'); ax.legend()
plt.tight_layout(); plt.show()
for r in rows:
    print(f"  {r['range (m)']:>6} m: AP@1m {r['AP@1m']:.3f}, AP@2m {r['AP@2m']:.3f}, mean centre error {r['mean centre error (m)']} m")
print(f"AP@1m falls from {rows[0]['AP@1m']:.2f} (near) to {rows[-1]['AP@1m']:.2f} (far): a single mAP hides this range dependence,"
      " which is exactly what a safety argument needs (cf. stopping distance in notebook 05).")

---
## 5. Dataset quality and diversity

**Why it matters for safety:** models learn only what they see. Rare scenarios are often missing, and geographic, weather and sensor biases affect generalisation.

In [ ]:
diversity = pd.DataFrame({
    'Dimension': ['Weather', 'Time of day', 'Geography', 'Sensors', 'Object classes (benchmark)'],
    'KITTI': ['Mostly fair', 'Day', 'Karlsruhe (DE)', 'Camera, LiDAR', '8 annotated, 3 evaluated'],
    'nuScenes': ['Incl. rain', 'Day & night', 'Boston, Singapore', 'Camera, LiDAR, radar', '23 annotated, 10 evaluated'],
    'Waymo Open': ['Mixed (see dataset paper)', 'Day & night', 'Several US cities', 'Camera, LiDAR', '4 main classes'],
    'BDD100K': ['Diverse', 'Day, dusk, night', 'US regions', 'Camera only', '10 detection classes (2D)'],
})
display(diversity)
for _, r in diversity.iterrows():
    print(f"  {r['Dimension']}: KITTI={r['KITTI']}; nuScenes={r['nuScenes']}; Waymo={r['Waymo Open']}; BDD100K={r['BDD100K']}")
print("\nDiversity gaps lead to safety issues:")
print("  1. Camera-centric systems can degrade in reduced visibility (cf. NHTSA PE24-031 on Tesla FSD in glare/fog/dust).")
print("  2. Systems may fail in new cities with different road layouts, signs and road users.")
print("  3. Rare objects (animals, debris, unusual vehicles) are under-represented and often missed.")
print("Mitigations: targeted data collection, augmentation, simulation and synthetic data - each with its own validity limits.")

> **Illustrative data:** the scores below are qualitative judgements on a 0–10 scale, chosen for teaching; they are not measurements.

In [ ]:
names_d = ['KITTI', 'nuScenes', 'Waymo', 'BDD100K']
scores = {'Weather': [3, 8, 6, 9], 'Time': [3, 8, 7, 8], 'Geography': [2, 6, 7, 8],
          'Traffic': [6, 8, 8, 9], 'Road types': [6, 7, 8, 9], 'Objects': [5, 9, 6, 7]}
ang = np.linspace(0, 2 * np.pi, len(scores), endpoint=False).tolist(); ang += ang[:1]
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
for i, d in enumerate(names_d):
    v = [scores[k][i] for k in scores] + [scores[list(scores)[0]][i]]
    ax.plot(ang, v, 'o-', lw=2, label=d, color=OKABE_ITO[[4, 0, 2, 5][i]]); ax.fill(ang, v, alpha=0.1, color=OKABE_ITO[[4, 0, 2, 5][i]])
ax.set_xticks(ang[:-1]); ax.set_xticklabels(list(scores)); ax.set_ylim(0, 10)
ax.set_title('Dataset diversity (illustrative qualitative scores, 0-10)', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout(); plt.show()
print("NOTE: values are illustrative teaching numbers, not measurements.")
tot = {d: sum(scores[k][i] for k in scores) for i, d in enumerate(names_d)}
for d, t_ in tot.items():
    weakest = min(scores, key=lambda k: scores[k][names_d.index(d)])
    print(f"  {d}: total {t_}/{10*len(scores)}, weakest dimension: {weakest}")
print("No single dataset covers every dimension -> combine sources and analyse coverage against your ODD.")

---
## 6. The long tail

Most driving is uneventful; safety-relevant scenarios are rare. A model trained on such data sees very few examples of exactly the situations that matter.

> **Illustrative data:** the scenario frequencies below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
scenarios = ['Highway cruising', 'Urban driving', 'Intersections', 'Parking lots', 'Night driving', 'Light rain',
             'Heavy rain', 'Construction zones', 'Emergency vehicles', 'Animals crossing', 'Debris on road',
             'Snow/ice', 'Fog', 'Unusual objects']
freq = np.array([100000, 80000, 60000, 40000, 20000, 10000, 3000, 2000, 1000, 500, 300, 200, 150, 50])
cat = np.where(freq > 10000, 'Common (>10K)', np.where(freq > 1000, 'Uncommon (1K-10K)', 'Rare (<=1K)'))
cmap = {'Common (>10K)': OKABE_ITO[2], 'Uncommon (1K-10K)': OKABE_ITO[0], 'Rare (<=1K)': OKABE_ITO[5]}
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(range(len(scenarios)), freq, color=[cmap[c] for c in cat])
ax.set_yscale('log'); ax.set_xticks(range(len(scenarios))); ax.set_xticklabels(scenarios, rotation=45, ha='right')
ax.set_ylabel('Occurrences in training data (log scale)')
ax.set_title('Long-tail scenario distribution (illustrative, not measured data)')
ax.legend(handles=[Patch(color=v, label=k) for k, v in cmap.items()])
plt.tight_layout(); plt.show()

print("NOTE: values are illustrative teaching numbers, not measurements.")
for s_, f_, c_ in zip(scenarios, freq, cat):
    print(f"  {s_:20s} {f_:>8,}  [{c_}]")
share_common = freq[cat == 'Common (>10K)'].sum() / freq.sum(); share_rare = freq[cat == 'Rare (<=1K)'].sum() / freq.sum()
print(f"Common scenarios: {share_common:.1%} of the data; rare scenarios: {share_rare:.2%}.")
print("Rare scenarios are under-represented in training data, yet a system must handle them safely"
      " (long tail / SOTIF 'unknown unsafe' scenarios). Targeted collection and simulation (e.g. CARLA) help fill gaps.")

---
## Exercise: dataset selection

**Scenario:** a perception system for an autonomous campus shuttle — urban, max 25 km/h, all seasons including snow, day and night; key risks are pedestrians, cyclists and unexpected obstacles.

Which dataset(s) would you choose, what gaps remain, and how would you close them? Also check the licences.

In [ ]:
your_choice = {
    'primary_dataset': '',          # e.g. 'nuScenes'
    'secondary_datasets': [],       # e.g. ['BDD100K', 'ZOD']
    'reasoning': '',
    'identified_gaps': [],
    'mitigation_strategy': '',
    'licence_check': '',            # commercial use allowed?
}
print("Consider: sensor coverage, speed range, weather (snow!), VRU density, licence terms, dataset size.")
# Sample answer (uncomment to see)
# sample_answer = {
#     'primary_dataset': 'nuScenes (360° camera+LiDAR+radar, night and rain, many pedestrians)',
#     'secondary_datasets': ['ZOD (European conditions)', 'BDD100K (weather/time diversity, camera only)'],
#     'identified_gaps': ['snow/ice', 'low-speed campus layouts', 'dense crowds', 'scooters, wheelchairs'],
#     'mitigation_strategy': 'Collect campus data across seasons, targeted labelling of rare VRUs, simulation for edge cases',
#     'licence_check': 'Most public datasets are non-commercial: a product needs own data or a commercial licence',
# }
# print(sample_answer)

---
## Safety perspective

- **ISO/PAS 8800:2024** treats data as a safety-relevant asset: datasets need specified requirements (coverage of the ODD, label quality, independence of training and test data) and verification — properties discussed throughout this notebook.
- **ISO 21448:2022:** gaps in the data show up as unknown triggering conditions (Area 3: unknown unsafe). Coverage analysis against the ODD (ISO 34503:2023) is a way to move scenarios from "unknown" to "known".
- **Metrics:** a single mAP or NDS hides range-, class- and condition-dependence; safety arguments need disaggregated metrics on held-out data from the target ODD.
- **Licences:** most public datasets are non-commercial; production development needs its own data and data management.

The author's work on simulation-based evaluation of 3D detectors in a SOTIF-related use case (Patel & Jung, VEHITS 2024) shows how synthetic LiDAR datasets can target specific triggering conditions that public datasets lack.

Continue with: [02_Failures_and_Robustness/02 — Corner cases](../02_Failures_and_Robustness/02_corner_cases.ipynb) · [04_SOTIF_ISO21448/04 — Simulation-based validation](../04_SOTIF_ISO21448/04_simulation_based_validation.ipynb) · [06_AI_Safety_ISO_PAS_8800/01 — AI safety standards](../06_AI_Safety_ISO_PAS_8800/01_ai_safety_standards.ipynb).

## Key takeaways

- Distinguish scenes, sensor frames and annotated keyframes: nuScenes has 1.4 M camera images but ~40 K annotated keyframes.
- KITTI (2012) remains a benchmark but has limited diversity and evaluates only Car/Pedestrian/Cyclist; nuScenes, Waymo Open, Argoverse 2 and the European ZOD and MAN TruckScenes add sensors, scale and conditions.
- Class imbalance is severe: VRU classes are a small fraction of KITTI labels yet dominate safety risk.
- nuScenes evaluates 3D detection by BEV centre distance (0.5/1/2/4 m) and combines mAP with TP errors into NDS; performance falls with range, which a single score hides.
- The long tail means safety-relevant scenarios are rare in data; targeted collection, augmentation and simulation are needed, each with validity limits.
- Most public datasets are licensed for non-commercial use only.
- ISO/PAS 8800 makes data requirements and dataset verification part of the safety case; ODD coverage gaps are SOTIF unknowns.

## Self-assessment

1. What are the main differences between KITTI and nuScenes in sensor configuration and geographic coverage?
2. Why does class imbalance pose a safety risk for perception?
3. Compute NDS for mAP = 0.40 and TP errors mATE 0.3 m, mASE 0.25, mAOE 0.5 rad, mAVE 0.8 m/s, mAAE 0.2.
4. Why can a detector have a good overall mAP but still be unsafe for a 50 km/h urban function?
5. You are deploying a shuttle in northern Europe. Which dataset(s) would you start from, and what gaps would you need to fill?
6. How can simulation and augmentation help address dataset limitations, and what new risk do they introduce?

<details>
<summary>Show answers</summary>

1. KITTI: front-facing stereo cameras and one 64-beam LiDAR, Karlsruhe only, mostly daytime fair weather. nuScenes: 6 cameras (360°), 5 radars and a 32-beam LiDAR; Boston and Singapore; includes rain and night.
2. Models see far fewer examples of rare classes (e.g. Person_sitting, cyclists, wheelchair users) and perform worse on them, while these are often the most vulnerable road users.
3. Sum of (1 − min(1, mTP)) = 0.7 + 0.75 + 0.5 + 0.2 + 0.8 = 2.95; NDS = 0.1 × (5 × 0.40 + 2.95) = 0.495.
4. mAP averages over classes, ranges and thresholds; the detector may be weak exactly on pedestrians within the stopping-distance range, or in night/rain conditions that are rare in the test set.
5. E.g. ZOD (European roads) plus nuScenes (360° sensors, night/rain); gaps: snow and ice, low sun, long nights, region-specific road users and signs — close them with own data collection, targeted labelling and simulation.
6. They generate rare scenarios cheaply and systematically (weather, lighting, rare objects). Risk: a domain gap between synthetic and real data — results must be validated on real data, and simulation validity must itself be argued.

</details>

## References

1. Geiger, A., Lenz, P., Urtasun, R. (2012). Are we ready for autonomous driving? The KITTI vision benchmark suite. CVPR. https://www.cvlibs.net/datasets/kitti/
2. Caesar, H. et al. (2020). nuScenes: A multimodal dataset for autonomous driving. CVPR. arXiv:1903.11027
3. Sun, P. et al. (2020). Scalability in Perception for Autonomous Driving: Waymo Open Dataset. CVPR. arXiv:1912.04838; https://waymo.com/open/about
4. Yu, F. et al. (2020). BDD100K: A Diverse Driving Dataset for Heterogeneous Multitask Learning. CVPR. arXiv:1805.04687
5. Wilson, B. et al. (2021). Argoverse 2: Next Generation Datasets for Self-Driving Perception and Forecasting. NeurIPS Datasets & Benchmarks. arXiv:2301.00493
6. Alibeigi, M. et al. (2023). Zenseact Open Dataset: A large-scale and diverse multimodal dataset for autonomous driving. ICCV. arXiv:2305.02008
7. Fent, F. et al. (2024). MAN TruckScenes: A multimodal dataset for autonomous trucking in diverse conditions. NeurIPS Datasets & Benchmarks. arXiv:2407.07462
8. Geyer, J. et al. (2020). A2D2: Audi Autonomous Driving Dataset. arXiv:2004.06320
9. ISO/PAS 8800:2024. *Road vehicles — Safety and artificial intelligence.*
10. ISO 21448:2022. *Road vehicles — Safety of the intended functionality.* https://www.iso.org/standard/77490.html
11. NHTSA ODI (2024). PE24-031 opening resume. https://static.nhtsa.gov/odi/inv/2024/INOA-PE24031-23232.pdf
12. Patel, M., Jung, R. (2024). Simulation-Based Performance Evaluation of 3D Object Detection Methods with Deep Learning for a LiDAR Point Cloud Dataset in a SOTIF-related Use Case. VEHITS 2024. https://doi.org/10.5220/0012707300003702

---
**Navigation:** ← [07 Multi-object tracking](07_multi_object_tracking.ipynb) · [Module index](README.md) · [Module 02 — Failures and Robustness](../02_Failures_and_Robustness/README.md) →